# finrag on Google Colab
Questions on company financial reports (10-K, 10-Q, 8-K, earnings), answered with retrieval + a 7B model and evaluated on the open FinanceBench sample.

Runtime → Change runtime type → **T4 GPU** before you start. Run the cells in order.
The language model is loaded **once** (step 4) and reused by every later cell: do not run the `python -m finrag ask/eval`
commands from the README in this notebook, they would load a second copy of the model.

In [ ]:
!nvidia-smi | head -15

## 1. Install and test
Replace the URL with your repository. The tests need no GPU and no download: they must all pass.
If the repository is already cloned in this session, run only `%cd finrag-repo` and `!git pull` to get the latest code.

In [ ]:
REPO_URL = "https://github.com/robertomodonesi94/financial-reports-rag"
%cd /content
!git clone $REPO_URL finrag-repo   # an "already exists" error is fine
%cd /content/finrag-repo
!git pull
!pip install -q -e ".[ml,api,dev]"
!pytest -q

## 2. Google Drive
Colab storage is wiped when the session ends: `data/` and the results are copied to Drive (`MyDrive/finrag`).
If you saved `data/` in a previous session, uncomment the `cp` line and skip step 3.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
SAVE = "/content/drive/MyDrive/finrag"
!mkdir -p $SAVE && ls $SAVE
# !cp -r $SAVE/data .     # <- uncomment to restore

## 3. Download the reports and build the index
`fetch` downloads the 150 FinanceBench questions and the PDFs they refer to (84 reports, a few minutes).
`ingest` reads every page, chunks it and embeds it on the GPU. Then save `data/` to Drive so you never rebuild it.

In [ ]:
!python -m finrag fetch
!python -m finrag ingest
!python -m finrag make-gold
!mkdir -p $SAVE && cp -r data $SAVE/

## 4. Load the pipeline once
The first run downloads the language model (about 14 GB, a few minutes): **check its licence** first.
Loading takes about a minute; afterwards each answer takes ~10-20 s.

In [ ]:
import sys
sys.path.insert(0, "src")   # import the code of this repository without restarting the runtime

import json
from finrag.config import Config
from finrag.pipeline import load_pipeline
from finrag.eval.dataset import load_dataset, sample, save_jsonl
from finrag.eval.metrics import number_match

CONFIG = "configs/default.yaml"
pipe = load_pipeline(Config.load(CONFIG))
rag, agent = pipe.rag(), pipe.agent()
print(len(pipe.index.chunks), "chunks from", len(pipe.docs), "reports")

def show(answer):
    """Print an answer with its sources as PDF page numbers (1-based, like your PDF viewer)."""
    print(answer.text)
    for s in answer.sources:
        print(f"  [{s['n']}] {s['doc_id']}  page {s['page'] + 1}")

## 5. Ask questions
Read every answer critically: open the cited page in the PDF (`data/pdf/`, file browser on the left) and check the number.
The 3M question is a real FinanceBench question: the expected answer is **USD 1,577 million**, on **page 60** of `3M_2018_10K.pdf`.

In [ ]:
Q = ("What is the FY2018 capital expenditure amount (in USD millions) for 3M? "
     "Give a response to the question by relying on the details shown in the cash flow statement.")
show(rag.ask(Q, doc_ids=["3M_2018_10K"]))   # only in the 3M report
print("=" * 80)
show(rag.ask(Q))                            # in all 84 reports

In [ ]:
res = agent.run("By how much did 3M's capital expenditure change between FY2017 and FY2018, in percent?\n"
                "(The relevant report: 3M_2018_10K.)")
print("ANSWER:", res.answer, "| finished:", res.finished)
for step in res.steps:
    print(json.dumps(step)[:400])

## 6. Block 3: ten FinanceBench questions
Ten random questions (fixed seed), each searched only in its report. For each one: expected answer, the model's answer,
gold pages vs retrieved pages, and whether the gold figure is in the answer. Check 2-3 of them in the PDF yourself.
Change `seed` for another ten.

In [ ]:
gold = load_dataset("data/eval/gold.jsonl")
for ex in sample(gold, 10, seed=0):
    a = rag.ask(ex["question"], doc_ids=ex["doc_ids"])
    gold_pages = sorted({p + 1 for _, p in ex["pages"]})
    found = [s["page"] + 1 for s in a.sources]
    print("QUESTION:", ex["question"][:220])
    print("EXPECTED:", ex["answer"][:220])
    print("ANSWER  :", a.text[:400])
    print("PAGES   : gold", gold_pages, "| retrieved", found, "| hit:", bool(set(gold_pages) & set(found)),
          "| right number:", number_match(a.text, ex["answer"]))
    print("-" * 100)

## 7. Evaluation
Before the final evaluation, add your ~8 unanswerable questions to `data/eval/gold.jsonl` (format in the README).
Each answer takes ~10-20 s: 40 questions x 2 variants is about 20-25 minutes. The results are saved and copied to Drive.

In [ ]:
from finrag.__main__ import VARIANTS
from finrag.eval.run import compare, to_markdown

examples = sample(load_dataset("data/eval/gold.jsonl"), 40)
chosen = {k: VARIANTS[k] for k in ["hybrid, shared", "hybrid, per_doc"]}   # also available: "dense, shared"
results = compare(examples, pipe, chosen, mode="rag")
print(to_markdown(results))
save_jsonl([{"variant": k, **v} for k, v in results.items()], "data/eval/results_rag.json")
!mkdir -p $SAVE/data/eval && cp data/eval/*.json* $SAVE/data/eval/

In [ ]:
results_agent = compare(sample(load_dataset("data/eval/gold.jsonl"), 20), pipe, {"hybrid, per_doc": VARIANTS["hybrid, per_doc"]}, mode="agent")
print(to_markdown(results_agent))
save_jsonl([{"variant": k, **v} for k, v in results_agent.items()], "data/eval/results_agent.json")
!cp data/eval/results_agent.json $SAVE/data/eval/

## 8. Look at the errors
Wrong figures and missing answers from the last RAG evaluation: the raw material for the error analysis in the README.
Text answers (no figure in the gold answer) cannot be checked automatically: the last line lists them, read them yourself.

In [ ]:
rows = results["hybrid, per_doc"]["rows"]
to_check = []
for r in rows:
    if r["type"] == "unanswerable":
        continue
    if r["abstained"] or r.get("number_match") is False:      # wrong figure or no answer
        print(r["id"], "| page hit:", r.get("page_hit"), "| abstained:", r["abstained"])
        print("  Q:", r["question"][:200])
        print("  expected:", r["gold"][:150])
        print("  answer  :", r["answer"][:250])
        print("  sources :", [f"{d} page {int(p) + 1}" for d, p in (s.rsplit(" p", 1) for s in r["sources"])])
    elif r.get("number_match") is None:                       # text answer: no automatic check possible
        to_check.append(r["id"])
print(len(to_check), "text answers to judge by hand:", to_check)
